In [0]:
# Load all DataFrames from Sparktables:
# Crime Data:
camb_df = spark.table('crime_data.silver.silver_cambridgeshire_crime').toPandas()
mers_df = spark.table('crime_data.silver.silver_merseyside_crime').toPandas()
nott_df = spark.table('crime_data.silver.silver_nottinghamshire_crime').toPandas()

# Extra Data:
adi_df = spark.table('crime_data.silver.silver_adi').toPandas()
houseprice_df = spark.table('crime_data.silver.silver_houseprice').toPandas()
#Do we have population data?
#Rural vs urban data?

In [0]:
# Aggregate the crime data:
camb_agg = camb_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')
mers_agg = mers_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')
nott_agg = nott_df.groupby(['year','month_num','force_name','lsoa_code','lsoa_name','crime_type'])['crime_type'].count().reset_index(name='total_crimes')

In [0]:
# Pivot the crime data:
camb_agg = camb_agg.pivot(index = ['year','month_num','force_name','lsoa_code','lsoa_name'], columns = 'crime_type', values = 'total_crimes').reset_index()
mers_agg = mers_agg.pivot(index = ['year','month_num','force_name','lsoa_code','lsoa_name'], columns = 'crime_type', values = 'total_crimes').reset_index()
nott_agg = nott_agg.pivot(index = ['year','month_num','force_name','lsoa_code','lsoa_name'], columns = 'crime_type', values = 'total_crimes').reset_index()


In [0]:
# Aggregate house price data
hp_agg = (
houseprice_df.groupby(["lsoa_code", "year", "month"], as_index=False)
      .agg(
          median_price=("price", "median"),
          transaction_count=("price", "size"),
      )
)
hp_agg["transaction_count"] = hp_agg["transaction_count"].astype("int64")

In [0]:
# Fill null values with 0:
camb_agg = camb_agg.fillna(0)
mers_agg = mers_agg.fillna(0)
nott_agg = nott_agg.fillna(0)


In [0]:
# Merge the crime data:
crime_agg = camb_agg.merge(mers_agg, on = ['year','month_num','force_name','lsoa_code','lsoa_name'], how = 'outer')

In [0]:
# Join the ADI and housing data
crime_agg = crime_agg.merge(hp_agg, left_on=['lsoa_code', 'year', 'month_num'], right_on=['lsoa_code', 'year', 'month'], how='left')
crime_agg = crime_agg.merge(adi_df, on=['lsoa_code'], how='left')

In [0]:
#Export to gold table
spark.createDataFrame(crime_agg).write.mode('overwrite').saveAsTable('crime_data.gold.gold_final_dataset')